# Does Window=60 Need More Training? (Convergence Check)

**Why this check exists**: comparing the last 20 epochs of each model's
training log:
- Window=30: val loss 0.2326 -> 0.2335 over epochs 281-300 (flat/plateaued).
- Window=60: val loss 0.5321 -> 0.5236 over epochs 281-300 (still steadily
  decreasing at the same rate as earlier in training).

Window=60 hit the 300-epoch cap while still improving, not because it
converged. This retrains it with `max_epochs=600` (same seed, same
hyperparameters, same data) to see (a) whether it actually converges given
more budget, and (b) whether that changes `cc1_test` and `drift_cc2`
results, in either direction.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, precision_recall_curve)

torch.manual_seed(42)
np.random.seed(42)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
OUT_DIR   = os.path.join(BASE, 'experiments')

DEVICE = torch.device('cpu')
ALL_SETS = ['cc1_test', 'drift_cc2']
HIDDEN1, HIDDEN2, LATENT_DIM, BETA_MAX = 64, 32, 32, 0.01
WARMUP_EPOCHS = 10
EXTENDED_MAX_EPOCHS, EXTENDED_PATIENCE = 600, 30   # doubled budget, slightly more patience
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0

print(f'Extended training: max_epochs={EXTENDED_MAX_EPOCHS}, patience={EXTENDED_PATIENCE}')

Extended training: max_epochs=600, patience=30


In [2]:
raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]
X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}')

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

print('Training with extended budget ...')
torch.manual_seed(42)
model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, LATENT_DIM).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
best_val, best_state, patience_ctr = float('inf'), None, 0
val_history = []

t0 = time.time()
for epoch in range(EXTENDED_MAX_EPOCHS):
    beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * BETA_MAX
    model.train()
    for (xb,) in loader:
        opt.zero_grad()
        recon, mu, logvar = model(xb)
        loss, rloss, kl = vae_loss(recon, xb, mu, logvar, beta)
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        recon, mu, logvar = model(X_val_t)
        vloss, vrecon, vkl = vae_loss(recon, X_val_t, mu, logvar, beta)
    val_history.append(vloss.item())
    if vloss.item() < best_val - 1e-6:
        best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        patience_ctr += 1
        if patience_ctr >= EXTENDED_PATIENCE:
            print(f'  early stop at epoch {epoch+1} (best val_loss={best_val:.4f})')
            break
    if epoch % 20 == 0 or epoch == EXTENDED_MAX_EPOCHS - 1:
        print(f'    epoch {epoch+1:3d}  beta={beta:.4f}  train_kl={kl.item():.4f}  val={vloss.item():.4f}')

elapsed = time.time() - t0
model.load_state_dict(best_state)
n_epochs_run = epoch + 1
print(f'\nDone in {elapsed/60:.1f} min.  epochs_run={n_epochs_run}  best_val_loss={best_val:.4f}')
print(f'Last 20 logged val losses: {[round(v,4) for v in val_history[-20:]]}')

INPUT_DIM=48
Training with extended budget ...
    epoch   1  beta=0.0010  train_kl=30.7573  val=1.1076
    epoch  21  beta=0.0100  train_kl=12.3409  val=0.7321
    epoch  41  beta=0.0100  train_kl=13.0444  val=0.6494
    epoch  61  beta=0.0100  train_kl=14.0857  val=0.6205
    epoch  81  beta=0.0100  train_kl=14.3178  val=0.6047
    epoch 101  beta=0.0100  train_kl=14.4914  val=0.5922
    epoch 121  beta=0.0100  train_kl=14.4992  val=0.5824
    epoch 141  beta=0.0100  train_kl=14.9093  val=0.5748
    epoch 161  beta=0.0100  train_kl=15.3099  val=0.5660
    epoch 181  beta=0.0100  train_kl=15.6983  val=0.5568
    epoch 201  beta=0.0100  train_kl=15.5882  val=0.5514
    epoch 221  beta=0.0100  train_kl=15.5934  val=0.5475
    epoch 241  beta=0.0100  train_kl=15.5732  val=0.5439
    epoch 261  beta=0.0100  train_kl=16.0710  val=0.5371
    epoch 281  beta=0.0100  train_kl=16.3225  val=0.5321
    epoch 301  beta=0.0100  train_kl=15.9870  val=0.5251
    epoch 321  beta=0.0100  train_kl=16.2

## Evaluation — extended-training model vs. the original 300-epoch model

In [3]:
with torch.no_grad():
    mse_train = model.anomaly_score(X_train_t).numpy()
    mse_val = model.anomaly_score(X_val_t).numpy()
mu_train, sigma_train = float(mse_train.mean()), float(mse_train.std())
val_p99 = float(np.percentile(mse_val, 99))
print(f'Extended model: mu_train={mu_train:.5f}  sigma_train={sigma_train:.5f}  val_p99={val_p99:.5f}')

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_pr': average_precision_score(y_true, scores), 'auc_roc': roc_auc_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

extended_results = {}
for name in ALL_SETS:
    X_t = torch.from_numpy(data[name])
    with torch.no_grad():
        mse = model.anomaly_score(X_t).numpy()
    extended_results[name] = evaluate(mse, labels[name], val_p99)

original_results = pickle.load(open(os.path.join(BASE, 'models_v2', 'vae_cc1_eval.pkl'), 'rb'))

print(f'\n{"set":12s} {"model":18s} {"PR-AUC":>8s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"OracleF1":>9s}')
for name in ALL_SETS:
    e = extended_results[name]
    o_auc = original_results['auc'][name]
    o_pr = original_results['precision_recall'][name]['val_p99']
    o_oracle = original_results['oracle_ceiling'][name]['f1']
    print(f'{name:12s} {"300 epochs (v2)":18s} {o_auc["auc_pr"]:8.4f} {o_pr["f1"]:7.3f} {o_pr["precision"]:10.3f} {o_pr["recall"]:8.3f} {o_oracle:9.4f}')
    print(f'{name:12s} {"600 epochs (ext)":18s} {e["auc_pr"]:8.4f} {e["f1"]:7.3f} {e["precision"]:10.3f} {e["recall"]:8.3f} {e["oracle_f1"]:9.4f}')
    print(f'  -> F1 change: {e["f1"]-o_pr["f1"]:+.3f}   Oracle-F1 change: {e["oracle_f1"]-o_oracle:+.4f}\n')

Extended model: mu_train=0.24455  sigma_train=0.22769  val_p99=1.37784

set          model                PR-AUC      F1  Precision   Recall  OracleF1
cc1_test     300 epochs (v2)      0.9178   0.912      0.986    0.848    0.9143
cc1_test     600 epochs (ext)     0.9343   0.925      0.991    0.867    0.9342
  -> F1 change: +0.013   Oracle-F1 change: +0.0199

drift_cc2    300 epochs (v2)      0.3307   0.155      0.086    0.800    0.3613
drift_cc2    600 epochs (ext)     0.3350   0.160      0.089    0.829    0.3513
  -> F1 change: +0.006   Oracle-F1 change: -0.0100



## Save

In [4]:
save_results = {
    'epochs_run': n_epochs_run, 'best_val_loss': best_val, 'val_history_tail': val_history[-20:],
    'mu_train': mu_train, 'sigma_train': sigma_train, 'val_p99': val_p99,
    'extended_results': extended_results,
}
out_path = os.path.join(OUT_DIR, 'window60_extended_training_check.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\window60_extended_training_check.pkl
